In [0]:
%sql
USE CATALOG workspace;
USE SCHEMA ecom_poc;

-- 1. Gold Daily Sales & Cancellation Rate (Grain: 1 row per order_date, currency)
CREATE OR REPLACE TABLE gold_daily_sales
COMMENT 'Daily sales performance and cancellation metrics at (order_date, currency) grain.'
AS
WITH order_payments AS (
  SELECT
    order_id,
    FIRST(currency) AS currency,
    SUM(CASE WHEN txn_status = 'SUCCESS' THEN
      CASE currency WHEN 'USD' THEN amount WHEN 'EUR' THEN ROUND(amount * 1.10, 2) WHEN 'GBP' THEN ROUND(amount * 1.27, 2) END
    ELSE 0 END) AS paid_amount_usd
  FROM workspace.ecom_poc.silver_transactions
  GROUP BY order_id
),
order_fact AS (
  SELECT
    o.order_id,
    o.order_date,
    COALESCE(p.currency, 'USD') AS currency,
    o.order_status,
    CASE WHEN o.order_status IN ('COMPLETED', 'SHIPPED') AND COALESCE(p.paid_amount_usd, 0) > 0
         THEN p.paid_amount_usd ELSE 0 END AS recognized_revenue_usd
  FROM workspace.ecom_poc.silver_orders o
  LEFT JOIN order_payments p ON o.order_id = p.order_id
)
SELECT
  order_date,
  currency,
  COUNT(DISTINCT order_id) AS total_orders,
  SUM(CASE WHEN order_status IN ('COMPLETED', 'SHIPPED') THEN 1 ELSE 0 END) AS fulfilled_orders,
  SUM(CASE WHEN order_status = 'CANCELLED' THEN 1 ELSE 0 END) AS cancelled_orders,
  ROUND(100.0 * SUM(CASE WHEN order_status = 'CANCELLED' THEN 1 ELSE 0 END) / COUNT(DISTINCT order_id), 2) AS cancellation_rate_pct,
  ROUND(SUM(recognized_revenue_usd), 2) AS recognized_revenue_usd
FROM order_fact
GROUP BY order_date, currency;

-- 2. Gold Revenue Breakdown by Category & Country (Grain: 1 row per category, country, currency)
CREATE OR REPLACE TABLE gold_revenue_breakdown
COMMENT 'Recognized revenue and order volume segmented by product category, country, and currency.'
AS
WITH order_payments AS (
  SELECT
    order_id,
    FIRST(currency) AS currency,
    SUM(CASE WHEN txn_status = 'SUCCESS' THEN
      CASE currency WHEN 'USD' THEN amount WHEN 'EUR' THEN ROUND(amount * 1.10, 2) WHEN 'GBP' THEN ROUND(amount * 1.27, 2) END
    ELSE 0 END) AS paid_amount_usd
  FROM workspace.ecom_poc.silver_transactions
  GROUP BY order_id
),
order_fact AS (
  SELECT
    o.order_id,
    o.category,
    o.country,
    o.quantity,
    COALESCE(p.currency, 'USD') AS currency,
    CASE WHEN o.order_status IN ('COMPLETED', 'SHIPPED') AND COALESCE(p.paid_amount_usd, 0) > 0
         THEN p.paid_amount_usd ELSE 0 END AS recognized_revenue_usd
  FROM workspace.ecom_poc.silver_orders o
  LEFT JOIN order_payments p ON o.order_id = p.order_id
)
SELECT
  category,
  country,
  currency,
  COUNT(DISTINCT order_id) AS total_orders,
  SUM(quantity) AS total_units_sold,
  ROUND(SUM(recognized_revenue_usd), 2) AS recognized_revenue_usd,
  ROUND(SUM(recognized_revenue_usd) / NULLIF(COUNT(DISTINCT order_id), 0), 2) AS avg_revenue_per_order_usd
FROM order_fact
GROUP BY category, country, currency;

-- 3. Gold Payment Success by Method (Grain: 1 row per payment_method, currency)
CREATE OR REPLACE TABLE gold_payment_success
COMMENT 'Payment gateway authorization volume and success rates at (payment_method, currency) grain.'
AS
SELECT
  payment_method,
  currency,
  COUNT(DISTINCT txn_id) AS total_attempts,
  SUM(CASE WHEN txn_status = 'SUCCESS' THEN 1 ELSE 0 END) AS successful_txns,
  SUM(CASE WHEN txn_status = 'FAILED' THEN 1 ELSE 0 END) AS failed_txns,
  SUM(CASE WHEN txn_status = 'PENDING' THEN 1 ELSE 0 END) AS pending_txns,
  ROUND(100.0 * SUM(CASE WHEN txn_status = 'SUCCESS' THEN 1 ELSE 0 END) / COUNT(DISTINCT txn_id), 2) AS success_rate_pct,
  ROUND(SUM(CASE WHEN txn_status = 'SUCCESS' THEN
    CASE currency WHEN 'USD' THEN amount WHEN 'EUR' THEN ROUND(amount * 1.10, 2) WHEN 'GBP' THEN ROUND(amount * 1.27, 2) END
  ELSE 0 END), 2) AS successful_volume_usd
FROM workspace.ecom_poc.silver_transactions
GROUP BY payment_method, currency;

-- 4. Gold Top Customers by Spend (Grain: 1 row per customer_id)
CREATE OR REPLACE TABLE gold_top_customers
COMMENT 'Customer lifetime recognized spend and order count ranked descending at customer_id grain.'
AS
WITH order_payments AS (
  SELECT
    order_id,
    SUM(CASE WHEN txn_status = 'SUCCESS' THEN
      CASE currency WHEN 'USD' THEN amount WHEN 'EUR' THEN ROUND(amount * 1.10, 2) WHEN 'GBP' THEN ROUND(amount * 1.27, 2) END
    ELSE 0 END) AS paid_amount_usd
  FROM workspace.ecom_poc.silver_transactions
  GROUP BY order_id
),
order_fact AS (
  SELECT
    o.order_id,
    o.customer_id,
    o.country,
    o.order_date,
    CASE WHEN o.order_status IN ('COMPLETED', 'SHIPPED') AND COALESCE(p.paid_amount_usd, 0) > 0
         THEN p.paid_amount_usd ELSE 0 END AS recognized_revenue_usd
  FROM workspace.ecom_poc.silver_orders o
  LEFT JOIN order_payments p ON o.order_id = p.order_id
)
SELECT
  customer_id,
  MAX(country) AS primary_country,
  COUNT(DISTINCT order_id) AS total_orders,
  ROUND(SUM(recognized_revenue_usd), 2) AS total_spend_usd,
  MAX(order_date) AS last_order_date,
  DENSE_RANK() OVER (ORDER BY SUM(recognized_revenue_usd) DESC) AS spend_rank
FROM order_fact
GROUP BY customer_id;

In [0]:
%sql
SELECT 'gold_daily_sales' AS table_name, COUNT(*) AS row_count, ROUND(SUM(recognized_revenue_usd), 2) AS total_revenue_usd FROM workspace.ecom_poc.gold_daily_sales
UNION ALL
SELECT 'gold_revenue_breakdown', COUNT(*), ROUND(SUM(recognized_revenue_usd), 2) FROM workspace.ecom_poc.gold_revenue_breakdown
UNION ALL
SELECT 'gold_top_customers', COUNT(*), ROUND(SUM(total_spend_usd), 2) FROM workspace.ecom_poc.gold_top_customers
UNION ALL
SELECT 'gold_payment_success', COUNT(*), ROUND(SUM(successful_volume_usd), 2) FROM workspace.ecom_poc.gold_payment_success;



In [0]:
%sql
SELECT * FROM workspace.ecom_poc.gold_top_customers ORDER BY spend_rank ASC LIMIT 10;